# Two-leg foundation-taker research test

This notebook turns the execution idea in `examples/foundation_taker` into a small, symmetric two-leg research test using the two CSV files in `data/`. The foundation example says that a causally available score may trigger an aggressive (taker-style) order at a policy-owned limit price. Here the score is a rolling z-score of the log mid-price spread between the two legs.

When the spread is unusually low, the test **buys leg 1 at its ask and sells leg 2 at its bid**. When it is unusually high, it does the reverse. It closes both legs when the spread returns near its rolling mean (or hits a stop). Signals formed on one snapshot execute on the following snapshot, so the test does not trade on information from the future.

> Scope: this is a compact research proxy, not the supported `ProductionReplayAdapter` route and not S0/economics evidence. It assumes top-of-book fills for one unit per leg, ignores depth/partial fills and latency beyond one row, and uses a configurable fee placeholder.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Works when launched from either the repository root or notebooks/.
repo = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "data").is_dir())
files = sorted((repo / "data").glob("market_data_2024-01-25_*.csv"))
assert len(files) == 2, f"Expected exactly two leg CSVs, found {files}"

def load_leg(path, suffix):
    cols = ["exchtime", "bidpx0", "bidvol0", "askpx0", "askvol0"]
    out = pd.read_csv(path, usecols=cols, parse_dates=["exchtime"])
    out = out.drop_duplicates("exchtime", keep="last").sort_values("exchtime")
    out = out.query("bidpx0 > 0 and askpx0 >= bidpx0 and bidvol0 > 0 and askvol0 > 0")
    return out.rename(columns={c: f"{c}_{suffix}" for c in cols if c != "exchtime"})

leg1 = load_leg(files[0], "1")
leg2 = load_leg(files[1], "2")
book = leg1.merge(leg2, on="exchtime", how="inner", validate="one_to_one").set_index("exchtime")
for n in (1, 2):
    book[f"mid_{n}"] = (book[f"bidpx0_{n}"] + book[f"askpx0_{n}"]) / 2

print(files[0].name, "<->", files[1].name)
print(f"{len(book):,} synchronized snapshots from {book.index.min()} to {book.index.max()}")
book[["mid_1", "mid_2"]].head()

## Causal signal and rules

The raw spread is `log(mid_1) - log(mid_2)`. Its z-score uses the previous `LOOKBACK` observations for its mean and standard deviation (`shift(1)`), while execution occurs one row after the signal. A negative extreme means leg 1 is relatively cheap; a positive extreme means leg 1 is relatively rich.

In [ ]:
LOOKBACK = 120       # about one minute for roughly 0.5-second snapshots
ENTRY_Z = 2.0
EXIT_Z = 0.25
STOP_Z = 4.0
FEE_PER_LEG = 0.0    # cash fee per one-leg execution; set this for the contract
MULTIPLIER = 1.0     # contract multiplier; set this for economic P&L

book["spread"] = np.log(book["mid_1"]) - np.log(book["mid_2"])
history = book["spread"].shift(1).rolling(LOOKBACK, min_periods=LOOKBACK)
book["zscore"] = (book["spread"] - history.mean()) / history.std(ddof=0)
book[["spread", "zscore"]].dropna().head()

In [ ]:
def run_two_leg_taker(df):
    """One-unit paired trades; direction +1 = long leg 1 / short leg 2."""
    trades, position = [], None
    clean = df.dropna(subset=["zscore"]).copy()

    for i in range(1, len(clean)):
        signal = clean.iloc[i - 1]  # signal known before this row
        row = clean.iloc[i]         # taker execution at this row's bid/ask
        ts = clean.index[i]

        if position is None:
            if i == len(clean) - 1:  # do not open a position that cannot be closed
                break
            direction = 1 if signal.zscore <= -ENTRY_Z else (-1 if signal.zscore >= ENTRY_Z else 0)
            if direction:
                position = {
                    "direction": direction, "entry_time": ts, "entry_z": signal.zscore,
                    "entry_1": row.askpx0_1 if direction == 1 else row.bidpx0_1,
                    "entry_2": row.bidpx0_2 if direction == 1 else row.askpx0_2,
                }
            continue

        crossed_mean = (position["direction"] == 1 and signal.zscore >= -EXIT_Z) or (position["direction"] == -1 and signal.zscore <= EXIT_Z)
        stopped = abs(signal.zscore) >= STOP_Z
        last_row = i == len(clean) - 1
        if crossed_mean or stopped or last_row:
            d = position["direction"]
            exit_1 = row.bidpx0_1 if d == 1 else row.askpx0_1
            exit_2 = row.askpx0_2 if d == 1 else row.bidpx0_2
            gross = d * ((exit_1 - position["entry_1"]) - (exit_2 - position["entry_2"])) * MULTIPLIER
            trades.append({**position, "exit_time": ts, "exit_z": signal.zscore,
                           "exit_1": exit_1, "exit_2": exit_2,
                           "exit_reason": "eod" if last_row else ("stop" if stopped else "mean_reversion"),
                           "gross_pnl": gross, "net_pnl": gross - 4 * FEE_PER_LEG})
            position = None

    result = pd.DataFrame(trades)
    if not result.empty:
        result["side"] = np.where(result.direction == 1, "long leg 1 / short leg 2", "short leg 1 / long leg 2")
        result["holding_seconds"] = (result.exit_time - result.entry_time).dt.total_seconds()
        result["equity"] = result.net_pnl.cumsum()
    return result

trades = run_two_leg_taker(book)
trades.head()

In [ ]:
if trades.empty:
    print("No trades. Lower ENTRY_Z or inspect the signal chart.")
else:
    summary = pd.Series({
        "trades": len(trades),
        "long_leg1_short_leg2": (trades.direction == 1).sum(),
        "short_leg1_long_leg2": (trades.direction == -1).sum(),
        "win_rate": (trades.net_pnl > 0).mean(),
        "total_net_pnl": trades.net_pnl.sum(),
        "average_net_pnl": trades.net_pnl.mean(),
        "worst_trade": trades.net_pnl.min(),
        "average_holding_seconds": trades.holding_seconds.mean(),
    })
    display(summary.to_frame("value"))
    display(trades.groupby("side").net_pnl.agg(["count", "sum", "mean"]))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)
book["zscore"].plot(ax=axes[0], lw=0.8, color="navy", title="Causal spread z-score")
for level, color in [(ENTRY_Z, "crimson"), (-ENTRY_Z, "crimson"), (EXIT_Z, "gray"), (-EXIT_Z, "gray")]:
    axes[0].axhline(level, color=color, ls="--", lw=0.8)
axes[0].set_ylabel("z-score")

if not trades.empty:
    trades.set_index("exit_time")["equity"].plot(ax=axes[1], drawstyle="steps-post", color="darkgreen", title="Realized net P&L (research proxy)")
else:
    axes[1].text(0.5, 0.5, "No completed trades", ha="center", va="center", transform=axes[1].transAxes)
axes[1].set_ylabel("P&L")
plt.tight_layout();

## Reading the result

Both directions use exactly the same rule; the `side` breakdown confirms whether the sample exercised each assignment. Before interpreting the P&L economically, set the real contract multiplier and fees and add exchange-specific latency, depth consumption, partial fills, session handling, and hedge-ratio sizing. Parameters above are illustrative and were not selected on a holdout set.